In [1]:
import wfdb
import numpy as np
import pathlib as pl

import requests


import matplotlib.pyplot as plt

import sys
sys.path.append('.')
sys.path.append('../')
sys.path.append('../classifier/')

# Looking at the wfdb interface
## Annotations / Symbols / Diagnoses

In [2]:
# Ok, we already have the st.petersburg dataset given from the physionet files
# Let's checkout what we have

# Helper functions
import os
from scipy.io import loadmat
from scipy import signal as sig

def load_challenge_data(filename):
    x = loadmat(filename)
    data = np.asarray(x['val'], dtype=np.float64)

    new_file = filename.replace('.mat', '.hea')
    input_header_file = os.path.join(new_file)
    with open(input_header_file, 'r') as f:
        header_data = f.readlines()
    return data, header_data


def resample(data, source_fs, target_fs):
    upsampling_factor = np.float(np.float(target_fs) / np.float(source_fs))

    out_length = int(data.shape[1] * upsampling_factor)
    out_data = np.empty((data.shape[0], out_length), dtype=data.dtype)

    for i_lead in range(0, data.shape[0]):
        out_data[i_lead, :] = sig.resample(data[i_lead, :], out_length)

    return out_data


def get_rmssd(rr_intervals):
    return np.sqrt(np.sum(np.square(rr_intervals-np.roll(rr_intervals, 1))[1:])/
                   float(len(rr_intervals)-1))/np.mean(rr_intervals)


def get_tpr(rr_intervals):
    ntpp = np.intersect1d(np.where(np.roll(rr_intervals, -1)>rr_intervals),
                          np.where(np.roll(rr_intervals, 1)>rr_intervals))
    ntpp = np.delete(ntpp, np.where(ntpp==0)[0])
    ntpp = np.delete(ntpp, np.where(ntpp==len(rr_intervals)-1)[0])
    ntpm = np.intersect1d(np.where(np.roll(rr_intervals, -1)<rr_intervals),
                          np.where(np.roll(rr_intervals, 1)<rr_intervals))
    ntpm = np.delete(ntpm, np.where(ntpm==0)[0])
    ntpm = np.delete(ntpm, np.where(ntpm==len(rr_intervals)-1)[0])    
    ntp = len(ntpp)+len(ntpm)
    return ntp/float(len(rr_intervals))


def get_entropy(rr_intervals, n_outliers=8, n_bins=16):
    rr_intervals = np.sort(rr_intervals)
    hist = np.histogram(rr_intervals[n_outliers:-n_outliers], bins=n_bins)[0]
    hist = hist[np.where(hist>0.)]
    p = hist/(len(rr_intervals)-2*n_outliers)
    entropy = -np.sum(p*np.log2(p))/np.log2(n_bins)
    return entropy



In [3]:
# Explore the dataset: What sampling rates do we have
petersburg_files = []

for file in pl.Path('../utils/temp/').iterdir():
    if file.name.split('.')[-1] == 'hea':
        # Open file an check sampling rate
        if file.name[0] == 'I':
            petersburg_files.append(file.name.split('.')[0])

In [4]:
# Ok, i do not want to read anything about the annoation file format, and wfdb just 

# ok this is the ugly way --> wfdb provides an alternative
def get_attr(name):
    # Format name
    physionet_name = "I" + name[-2:]
    link = "https://physionet.org/files/incartdb/1.0.0/" + physionet_name + ".atr?download"
    with requests.Session() as s:
        download = s.get(link)
        decoded_content = download.content
        with open('I0001.atr', 'wb') as f:
            f.write(decoded_content)
    anno = wfdb.rdann('I0001', 'atr', sampto = 1000)
    
    return anno

#annotation = get_attr('I0002')
annotation = wfdb.rdann('I02', 'atr', pn_dir='incartdb')

#sorted(petersburg_files)
#https://physionet.org/files/incartdb/1.0.0/I01.atr?download

In [5]:
b = wfdb.show_ann_labels()

    label_store symbol                                    description
0             0                              Not an actual annotation
1             1      N                                    Normal beat
2             2      L                  Left bundle branch block beat
3             3      R                 Right bundle branch block beat
4             4      a                Aberrated atrial premature beat
5             5      V              Premature ventricular contraction
6             6      F          Fusion of ventricular and normal beat
7             7      J              Nodal (junctional) premature beat
8             8      A                   Atrial premature contraction
9             9      S     Premature or ectopic supraventricular beat
10           10      E                        Ventricular escape beat
11           11      j                 Nodal (junctional) escape beat
12           12      /                                     Paced beat
13           13     

In [ ]:
annos_unique = []
# Check out all the labels
for file in pl.Path('../utils/temp/').iterdir():
    if file.name.split('.')[-1] == 'hea':
        # Open file an check sampling rate
        if file.name[0] == 'I':
            physionet_name = "I" + file.name.replace('.hea', '')[-2:]
            annotation = wfdb.rdann(physionet_name, 'atr', pn_dir='incartdb')
            annos_unique.append(annotation.symbol)


In [ ]:
lu, lcnt = np.unique(np.concatenate(annos_unique).ravel(), return_counts=True)
lu, lcnt
# Annotation symbols from the original dataset
#'+', Rhythm change
#'A', Atrial premature contraction
#'B', Left or right bundle branch block
#'F', Fusion of ventricular and normal beat
#'N', Normal beat
#'Q', Unclassifiable beat
#'R', Right bundle branch block beat
#'S', Premature or ectopic supraventricular beat
#'V', Premature ventricular contraction
#'j', Nodal (junctional) escape beat
#'n', Supraventricular escape beat

In [ ]:
a = """1             1      N                                    Normal beat
2             2      L                  Left bundle branch block beat
3             3      R                 Right bundle branch block beat
4             4      a                Aberrated atrial premature beat
5             5      V              Premature ventricular contraction
6             6      F          Fusion of ventricular and normal beat
7             7      J              Nodal (junctional) premature beat
8             8      A                   Atrial premature contraction
9             9      S     Premature or ectopic supraventricular beat
10           10      E                        Ventricular escape beat
11           11      j                 Nodal (junctional) escape beat
12           12      /                                     Paced beat
13           13      Q                            Unclassifiable beat
14           14      ~                          Signal quality change
16           16      |                     Isolated QRS-like artifact
18           18      s                                      ST change
19           19      T                                  T-wave change
20           20      *                                        Systole
21           21      D                                       Diastole
22           22      "                             Comment annotation
23           23      =                         Measurement annotation
24           24      p                                    P-wave peak
25           25      B              Left or right bundle branch block
26           26      ^                      Non-conducted pacer spike
27           27      t                                    T-wave peak
28           28      +                                  Rhythm change
29           29      u                                    U-wave peak
30           30      ?                                       Learning
31           31      !                       Ventricular flutter wave
32           32      [      Start of ventricular flutter/fibrillation
33           33      ]        End of ventricular flutter/fibrillation
34           34      e                             Atrial escape beat
35           35      n                   Supraventricular escape beat
36           36      @  Link to external data (aux_note contains URL)
37           37      x             Non-conducted P-wave (blocked APB)
38           38      f                Fusion of paced and normal beat
39           39      (                                 Waveform onset
40           40      )                                   Waveform end
41           41      r       R-on-T premature ventricular contraction"""
lbls = {}
for i in a.split("\n"):
    s = " ".join(i.split()[2:])
    lbls[s[0]] = s[2:]

for s in lu:
    print(s, lbls[s])

# Automatic Merge Possible?

In [ ]:
# Manually search in scored concepts for matching paris

#SNOMEDID LBL NAME
# + Rhythm change                              ---> ?
# A Atrial premature contraction               ---> 284470004 (scored)
# B Left or right bundle branch block          ---> 59118001 (right) or 164909002 (left)
# F Fusion of ventricular and normal beat      ---> 13640000
# N Normal beat                                ---> 426783006 (scored, sinus rhythm and norm merged)
# Q Unclassifiable beat                        ---> ?
# R Right bundle branch block beat             ---> 59118001 (scored)
# S Premature or ectopic supraventricular beat ---> 63593006 (scored)
# V Premature ventricular contraction          ---> 427172004 (scored)
# j Nodal (junctional) escape beat             ---> 426995002 (unscored)
# n Supraventricular escape beat               ---> 75532003 (unscored)

## For some arrhythimas it may be possible to do it beat by beat, however this does not seam resonable...
anno_snomed_map = {
    "+" : 0,
    "A" : 284470004,
    "B" : 164909002,
    "F" : 13640000,
    "N" : 426783006,
    "Q" : 0,
    "R" : 59118001,
    "S" : 63593006,
    "V" : 427172004,
    "j" : 426995002,
    "n" : 75532003
}
anno_text_map = {
    "+" :"Rhythm change",
    "A" :"Atrial premature contraction",
    "B" :"Left or right bundle branch block",
    "F" :"Fusion of ventricular and normal beat",
    "N" :"Normal beat",
    "Q" :"Unclassifiable beat",
    "R" :"Right bundle branch block beat",
    "S" :"Premature or ectopic supraventricular beat",
    "V" :"Premature ventricular contraction",
    "j" :"Nodal (junctional) escape beat",
    "n" : "Supraventricular escape beat"
}
anno_snomed_map, anno_text_map

In [ ]:
# Small helper code to translate the snomeds
scored_file = pl.Path("../evaluation/dx_mapping_scored.csv").resolve()
unscored_file = pl.Path("../evaluation/dx_mapping_unscored.csv").resolve()
import csv

snomed_text_map = {}
snomed_abbr_map = {}
with open(scored_file) as csvfile:
    for row in csvfile.read().split("\n")[1:-1]:
        text, snomed, abbr = row.split(",")[:3]
        snomed_text_map[int(snomed)] = text
        snomed_abbr_map[int(snomed)] = abbr

with open(unscored_file) as csvfile:
    for row in csvfile.read().split("\n")[1:-1]:
        text, snomed, abbr = row.split(",")[:3]
        snomed_text_map[int(snomed)] = text
        snomed_abbr_map[int(snomed)] = abbr


In [ ]:
# Let's build a map to .... maybe - maybe .... find correlation 
# between challenge labels, physionet labels and annotations

def print_lables(physio_diag, physio_anno, challenge_lbls):
    print("PHYSIO Diagnoses:")
    for diag in physio_diag:
        print("\t", diag)
    print("PHYSIO Annotations:")
    for diag in physio_anno:
        print("\t", diag)
    print("Challenge Diagnoses:")
    for diag in challenge_lbls:
        print("\t", diag)

def get_diagnose_from_wfdb(name):
    head_original = wfdb.rdheader(name, pn_dir='incartdb')
    # Find diagnoses from free - text annoations
    infos = ("___").join(head_original.__dict__['comments']).\
        split("<sex>:")[1][2:].replace("<diagnoses>", "").strip().split("___")

    # Unsorted list (find rule of thumb for easy filtering, without regular expressions)
    patient = ""
    diagnose = []
    for info in infos:
        if info[:len("patient")] == "patient":
            patient = info
            continue
        if len(info) == 0:
            continue
        diagnose.append(info)
    free_text = [i.strip() for i in ",".join(diagnose).replace("and", ",").split(",")]
    return free_text, patient

def load_challenge_labels(name):
    with open(file) as f:
        head_challenge = f.read().split('\n')[-5].split(' ')[-1].split(',')
    head_challenge = [int(i) for i in head_challenge]

    return [(i, snomed_text_map[i], snomed_abbr_map[i]) for i in head_challenge]


In [ ]:
for file in pl.Path('../utils/temp/').iterdir():
    if file.name.split('.')[-1] == 'hea':
        # Open file an check sampling rate
        if file.name[0] == 'I':
            # Get the header from the challenge dataset
            with open(file) as f:
                head_challenge = f.read().split('\n')[-5].split(' ')[-1].split(',')
            head_challenge = [int(i) for i in head_challenge]

            # Get the header from the original dataset
            physionet_name = "I" + file.name.replace('.hea', '')[-2:]

            # Now check if the annotations match the classes in the annotation file
            annotation = wfdb.rdann(physionet_name, 'atr', pn_dir='incartdb')
            head_annos = np.unique(np.array(annotation.symbol))
            
            physionet_diagnose, patient = get_diagnose_from_wfdb(physionet_name)
            
            print('#' * 25, file.name, '#' * 25)
            print_lables(physionet_diagnose,
                         [(anno_snomed_map[i], i, anno_text_map[i]) for i in head_annos],
                         [(i, snomed_text_map[i], snomed_abbr_map[i]) for i in head_challenge])

# Examples: Looking at the Data

In [ ]:
record = wfdb.rdrecord('I03', sampto=10000, pn_dir='incartdb')
annotation = wfdb.rdann('I03', 'atr', sampto=10000, pn_dir='incartdb')
head_original = wfdb.rdheader('I03', pn_dir='incartdb')

physionet_diagnose, patient = get_diagnose_from_wfdb("I03")

print(annotation.symbol)
print(physionet_diagnose)

wfdb.plot_wfdb(record=record, annotation=annotation, time_units='seconds', figsize=(20, 20))

## Conclusion: Even if the labels / diagnoses have Acute MI, ST elevation, PVCs, every beat is classified as normal
##  in this segment. Well that was expected, but how do we map the diagnoses to the segments?

In [ ]:
record = wfdb.rdrecord('I04', sampfrom=10000, sampto=60000, pn_dir='incartdb')
annotation = wfdb.rdann('I04', 'atr', sampfrom=10000, sampto=60000, pn_dir='incartdb', shift_samps=True)
physionet_diagnose, patient = get_diagnose_from_wfdb("I04")

print(annotation.symbol)
print(physionet_diagnose)

wfdb.plot_wfdb(record=record, annotation=annotation, time_units='seconds', figsize=(20, 20))

## Conclusion: Even if the labels / diagnoses have Acute MI, ST elevation, PVCs, every beat is classified as normal
##  in this segment. Well that was expected, but how do we map the diagnoses to the segments?

In [ ]:
# Our base is 500Hz
fs = annotation.fs
samples = (annotation.sample / np.float(fs)) * 500.
symbols = annotation.symbol

In [ ]:
max(samples)/10000

# Write Function that does a mapping

This function is all but perfect, see analysis above, however it is something

In [ ]:
import wfdb
import numpy as np
import pathlib as pl

def compute_hr(annotations):
    rr = [annotations[i + 1] - annotations[i] for i in range(0, len(annotations)-1)]
    hr = np.mean(60/(np.array(rr)/500.))
    return hr

def get_diagnose_from_wfdb(name):
    head_original = wfdb.rdheader(name, pn_dir='incartdb')
    # Find diagnoses from free - text annoations
    infos = ("___").join(head_original.__dict__['comments']).\
        split("<sex>:")[1][2:].replace("<diagnoses>", "").strip().split("___")

    patient = ""
    diagnose = []
    for info in infos:
        if info[:len("patient")] == "patient":
            patient = info
            continue
        if len(info) == 0:
            continue
        diagnose.append(info)
    free_text = [i.strip() for i in ",".join(diagnose).replace("and", ",").split(",")]
    return free_text, patient

def petersburg_split_and_relabel(name, ecg_500Hz, challenge_labels):
    ecgs, labels = [], []

    # Load physionet labels
    physionet_name = "I" + name.replace('.hea', '')[-2:]
    physionet_labels, _ = get_diagnose_from_wfdb(physionet_name)
    
    # Load physionet annotations
    annotation = wfdb.rdann(physionet_name, 'atr', pn_dir='incartdb')
    
    # Resample annotations to 500 Hz
    samples = (annotation.sample / np.float(annotation.fs)) * 500.
    symbols = np.array(annotation.symbol)

    # Split ecg and annotations (each a length of 2 ** 13 + 2 ** 11 = 10240)
    for i in range(0, ecg_500Hz.shape[1], 10240):
        beg = i
        end = i + 10240 if i < ecg.shape[1] else ecg_500Hz.shape[1]
        
        
        ecg_slice = ecg_500Hz[:, beg:end]
        sl = (samples > float(beg)) & (samples < float(end))
        ann_slice_samples = samples[sl]
        ann_slice_symbols = symbols[sl]

        slow_fast_rhythms = [164889003, 164890007, 426627000, 427084000, 426177001]
        
        label = []
        # Everything looks normal
        if len(np.unique(ann_slice_symbols)) == 1 and  np.unique(ann_slice_symbols)[0] == "N":
            # Add normal class (sinus-rhythm and normal are merged)
            label.append(426783006)
        # Otherwise add the labels from the challenge (except tachycardia/bradycardia/etc..)
        else:
            challenge_labels_snomeds = set(challenge_labels)
            for sfr in slow_fast_rhythms:
                challenge_labels_snomeds.discard(sfr)
            for lbl in challenge_labels_snomeds:
                label.append(lbl)

        # Special Case: Check if tachycardia / bradicardia / flutter / fibrilation
        # Apply a simple rule based mechanism for rule based rhythms
        # tachycardia >= 120 bpm
        # bradycardia <= 50 bpbm
        
        challenge_labels_snomeds = challenge_labels
        
        if not len(np.intersect1d(slow_fast_rhythms, challenge_labels_snomeds)):
            # We only add brady / tachy / flutter / fibrilation to the labels if they are annotated as such
            labels.append(label)
            ecgs.append(ecg_slice)
            continue
                
        # atrial fibrilations or flutter
        if 164889003 in challenge_labels_snomeds:
            label.append(164889003)
        if 164890007 in challenge_labels_snomeds:
            label.append(164890007)
            
            # No special rule, all sliced records contain afibs if label is present
            #print("Atrial Fibriations / Flutter",
            #      label,
            #      get_entropy(ann_slice_samples),
            #      get_rmssd(ann_slice_samples),
            #      get_entropy(ann_slice_samples))
        
        # Bradycardia
        if 426627000 in challenge_labels_snomeds:
            # Brady: hr < 60 bpm
            hr = compute_hr(ann_slice_samples)
            if hr < 60:
                label.append(426627000)
        
        if 426177001 in challenge_labels_snomeds:
            # Brady: hr < 60 bpm
            hr = compute_hr(ann_slice_samples)
            if hr < 60:
                label.append(426177001)

        # Tachycardia
        if 427084000 in challenge_labels_snomeds:
            hr = compute_hr(ann_slice_samples)
            # Tachy: hr > 100 bpm
            if hr > 100.:
                label.append(427084000)
        
        labels.append(label)
        ecgs.append(ecg_slice)

    return ecgs, labels

In [ ]:
# Helper functions
import os
from scipy.io import loadmat
from scipy import signal as sig

def load_challenge_data(filename):
    x = loadmat(filename)
    data = np.asarray(x['val'], dtype=np.float64)

    new_file = filename.replace('.mat', '.hea')
    input_header_file = os.path.join(new_file)
    with open(input_header_file, 'r') as f:
        head_challenge = f.read().split('\n')[-5].split(' ')[-1].split(',')
    head_challenge = [int(i) for i in head_challenge]

    return data, head_challenge


def resample(data, source_fs, target_fs):
    upsampling_factor = np.float(np.float(target_fs) / np.float(source_fs))

    out_length = int(data.shape[1] * upsampling_factor)
    out_data = np.empty((data.shape[0], out_length), dtype=data.dtype)

    for i_lead in range(0, data.shape[0]):
        out_data[i_lead, :] = sig.resample(data[i_lead, :], out_length)

    return out_data

fecgs, flabels = [], []
for file in pl.Path('../utils/temp/').iterdir():
    if file.name.split('.')[-1] == 'hea':
        # Open file an check sampling rate
        if file.name[0] == 'I':
            matfile = str(file.resolve().absolute())[:-4] + '.mat'
            ecg, lbls = load_challenge_data(matfile)
            
            # Resample 
            ecg = resample(ecg, 257, 500)
            # Load challange labels

            ecgs, labels = petersburg_split_and_relabel(file.name, ecg, lbls)
            
            fecgs.append(ecgs)
            flabels.append(labels)